## 🎬 Sistema de Recomendación de Películas — Modelo Baseline

En este proyecto se desarrolla un sistema de recomendación de películas basado en el dataset MovieLens. El objetivo es generar sugerencias personalizadas o generales a partir del historial de valoraciones de los usuarios.

El flujo del proyecto se divide en dos fases principales:

- Ingeniería de datos: limpieza, validación y preparación de los datos mediante un pipeline (ingest.py).
- Modelado: desarrollo de algoritmos de recomendación a partir de los datos procesados.

En este notebook se implementa un modelo baseline basado en popularidad, que sirve como punto de partida para sistemas más avanzados.

Este modelo recomienda las películas mejor valoradas globalmente por los usuarios, aplicando ciertos filtros para evitar ruido en los datos.

## Bloque 1: Importación de CSV

In [4]:
import pandas as pd

# Cargar datos procesados
ratings = pd.read_csv("../../../data/processed/ratings_sample.csv")
movies = pd.read_csv("../../../data/processed/movies_clean.csv")

# Vista inicial
print("Ratings:")
display(ratings.head())

print("\nMovies:")
display(movies.head())

Ratings:


,userId,movieId,rating,timestamp
0,33183,21,5.0,942800860
1,27473,1375,3.0,973042848
2,160083,4888,2.0,1008171230
3,96819,122906,4.5,1688662566
4,79969,6377,4.0,1395785884



Movies:


,movieId,title,genres
0,1,Toy Story (1995),"['Adventure', 'Animation', 'Children', 'Comedy..."
1,2,Jumanji (1995),"['Adventure', 'Children', 'Fantasy']"
2,3,Grumpier Old Men (1995),"['Comedy', 'Romance']"
3,4,Waiting to Exhale (1995),"['Comedy', 'Drama', 'Romance']"
4,5,Father of the Bride Part II (1995),['Comedy']


## Bloque 2: Exploración básica

In [5]:
print("Informacion del dataset:")
ratings.info()

print("\nEstadísticas:")
display(ratings.describe())

print("\nNúmero de usuarios:", ratings["userId"].nunique())
print("Número de películas:", ratings["movieId"].nunique())

Informacion del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100000 non-null  int64  
 1   movieId    100000 non-null  int64  
 2   rating     100000 non-null  float64
 3   timestamp  100000 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB

Estadísticas:


,userId,movieId,rating,timestamp
count,100000.000000,100000.000000,100000.000000,1.000000e+05
mean,100254.730870,28941.737420,3.542735,1.275143e+09
std,57867.729633,50316.851064,1.058210,2.558518e+08
min,9.000000,1.000000,0.500000,8.228736e+08
25%,50254.500000,1238.000000,3.000000,1.051534e+09
50%,100038.000000,3471.000000,3.500000,1.272314e+09
75%,150276.000000,44022.000000,4.000000,1.503158e+09
max,200944.000000,290573.000000,5.000000,1.697131e+09



Número de usuarios: 58888
Número de películas: 10913


## Bloque 3: Modelo baseline (popularidad)

In [6]:
# Calcular media de rating por película

top_movies = (
    ratings.groupby("movieId")["rating"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Añadir títulos
top_movies = top_movies.merge(movies, on="movieId")

# Mostrar top 10
display(top_movies[["title", "rating"]].head(10))

,title,rating
0,All the Beauty and the Bloodshed (2022),5.0
1,Love Again (2023),5.0
2,HyperNormalisation (2016),5.0
3,Russell Peters: Almost Famous (2016),5.0
4,The Siege of Jadotville (2016),5.0
5,My Life as a Courgette (2016),5.0
6,Goldstone (2016),5.0
7,Absolutely Fabulous: The Movie (2016),5.0
8,AC/DC- Let There Be Rock (1980),5.0
9,The Long Recess (1972),5.0


## Bloque 4: Mejora del modelo con mínimo de valoraciones

In [7]:
# Número de ratings por película
ratings_count = ratings.groupby("movieId")["rating"].count()

# Filtrar películas con suficientes valoraciones
min_ratings = 20
valid_movies = ratings_count[ratings_count >= min_ratings].index

filtered_ratings = ratings[ratings["movieId"].isin(valid_movies)]

# Recalcular ranking
top_movies_filtered = (
    filtered_ratings.groupby("movieId")["rating"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .merge(movies, on="movieId")
)

# Mostrar resultados
display(top_movies_filtered[["title", "rating"]].head(10))

,title,rating
0,Crimes and Misdemeanors (1989),4.425000
1,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",4.400000
2,"Shawshank Redemption, The (1994)",4.385220
3,"Godfather, The (1972)",4.381773
4,Parasite (2019),4.378788
5,Intouchables (2011),4.358333
6,"Godfather: Part II, The (1974)",4.319149
7,Spirited Away (Sen to Chihiro no kamikakushi) ...,4.315476
8,To Kill a Mockingbird (1962),4.307692
9,Rear Window (1954),4.296512


## Bloque 5: Funcion de recomendación

In [8]:
def recommend_top_movies(df, n=10):
    return df.sort_values("rating", ascending=False).head(n)[["title", "rating"]]

# Ejemplo
recommendations = recommend_top_movies(top_movies_filtered, 10)
display(recommendations)

,title,rating
0,Crimes and Misdemeanors (1989),4.425000
1,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",4.400000
2,"Shawshank Redemption, The (1994)",4.385220
3,"Godfather, The (1972)",4.381773
4,Parasite (2019),4.378788
5,Intouchables (2011),4.358333
6,"Godfather: Part II, The (1974)",4.319149
7,Spirited Away (Sen to Chihiro no kamikakushi) ...,4.315476
8,To Kill a Mockingbird (1962),4.307692
9,Rear Window (1954),4.296512
